In [1]:
""""
Copyright (C) 2025 ETH Zurich, Stefan J.P. Chrin, Gregory Landrum, Sereina Riniker
"""


import os
import pandas as pd
import numpy as np
import glob
import json

# ------ Paths ------
molecules_csv_path = "../../../0_DATA/COMPOUNDS/0_MOLECULES.csv"
base_dir = "./A_QSAR_MIF"
out_dir = "./B_FORMAT_PREDICTIONS"

# ----- Load molecules CSV -----
molecules_df = pd.read_csv(molecules_csv_path)

predictions = []  # list to store each prediction row
experiment_id = 1  # init experiment counter
prediction_id = 1  # init prediction counter

# ----- Iterate over all PLS_Analysis folders (new file structure) -----
paths = sorted(glob.glob(os.path.join(base_dir, '*', 'PLS_Analysis')))

for path in paths:
    # get target from parent folder
    target = os.path.normpath(path).split(os.sep)[-2]

    # iterate all files in PLS_Analysis
    for file in sorted(os.listdir(path)):
        # ensure file name has 5 underscores
        if file.count('_') != 5:

            raise ValueError(f"File {file} in path {path} does not match expected format (unwanted file in directory).")

        file_path = os.path.join(path, file)
        if os.path.isfile(file_path):
            parts = file.split("_")

            # extract vars from filename
            resolution = parts[3]
            variance_cutoff = parts[5].removesuffix('.json')

            # Load JSON file 
            with open(file_path, 'r') as f:
                data_dict = json.load(f)
                data_predictions = data_dict["split_predictions"]

            # Iterate through each prediction 
            for pred in data_predictions:
                row_dict = pred.copy()
                row_dict.pop("Set", None)  # remove unwanted column
                row_dict["per_target_index"] = row_dict.pop("Per target Index (X)")
                row_dict["target"] = target
                row_dict["resolution"] = resolution
                row_dict["folder"] = file_path
                row_dict["prediction_id"] = prediction_id
                row_dict["variance_cutoff"] = variance_cutoff

                #Lookup molecule info to put in df
                match = molecules_df[
                    (molecules_df['target'] == target) &
                    (molecules_df['per_target_index'] == int(row_dict["per_target_index"]))
                ]

                if match.empty:
                    raise Exception(f"Row not found in molecules CSV: {target} {row_dict['per_target_index']}")
                else:
                    match_row = match.iloc[0]
                    row_dict["smiles"] = match_row.get("smiles")
                    row_dict["molregno"] = match_row.get("molregno")
                    row_dict["ligand"] = match_row.get("ligand")
                    row_dict["sdf_dG_path"] = match_row.get("sdf_dG_path")

                predictions.append(row_dict)
                prediction_id += 1

# Create df from predictions 
predictions_df = pd.DataFrame(predictions)
predictions_df




,split,Actual,Predicted,Residual,per_target_index,target,resolution,folder,prediction_id,variance_cutoff,smiles,molregno,ligand,sdf_dG_path
0,0,-9.67,-8.513844,1.156156,35,bace,1.0,./A_QSAR_MIF/bace/PLS_Analysis/CV_Results_res_...,1,0.001,CN1C(=O)[C@@](c2ccccc2)(c2cccc(-c3cncc(Cl)c3)c...,286,CAT-4p,0_DATA/COMPOUNDS/SDF/bace_lig_CAT-4p.sdf
1,0,-8.13,-7.932345,0.197655,13,bace,1.0,./A_QSAR_MIF/bace/PLS_Analysis/CV_Results_res_...,2,0.001,CN1C(=O)[C@](c2cccc(-c3cccc(Cl)c3)c2)(c2nccs2)...,264,CAT-13o,0_DATA/COMPOUNDS/SDF/bace_lig_CAT-13o.sdf
2,0,-7.44,-8.333633,-0.893633,26,bace,1.0,./A_QSAR_MIF/bace/PLS_Analysis/CV_Results_res_...,3,0.001,COc1ccc(-c2cccc([C@@]3(c4ccccc4)[NH+]=C(N)N(C)...,277,CAT-4c,0_DATA/COMPOUNDS/SDF/bace_lig_CAT-4c.sdf
3,0,-7.44,-8.334266,-0.894266,30,bace,1.0,./A_QSAR_MIF/bace/PLS_Analysis/CV_Results_res_...,4,0.001,CN1C(=O)[C@@](c2ccccc2)(c2cccc(-c3ccncc3)c2)[N...,281,CAT-4k,0_DATA/COMPOUNDS/SDF/bace_lig_CAT-4k.sdf
4,0,-9.45,-9.492779,-0.042779,16,bace,1.0,./A_QSAR_MIF/bace/PLS_Analysis/CV_Results_res_...,5,0.001,CN1C(=O)[C@](c2cccc(-c3cccc(C#N)c3)c2)(C2CC2)[...,267,CAT-17c,0_DATA/COMPOUNDS/SDF/bace_lig_CAT-17c.sdf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12195,98,-9.63,-10.188071,-0.558071,0,tyk2,1.0,./A_QSAR_MIF/tyk2/PLS_Analysis/CV_Results_res_...,12196,0.001,CC(=O)Nc1cc(NC(=O)c2c(Cl)cccc2Cl)ccn1,315,ejm_31,0_DATA/COMPOUNDS/SDF/tyk2_lig_ejm_31.sdf
12196,99,-11.38,-11.337808,0.042192,13,tyk2,1.0,./A_QSAR_MIF/tyk2/PLS_Analysis/CV_Results_res_...,12197,0.001,O=C(Nc1ccnc(NC(=O)[C@H]2C[C@H]2Cl)c1)c1c(Cl)cc...,328,jmc_27,0_DATA/COMPOUNDS/SDF/tyk2_lig_jmc_27.sdf
12197,99,-9.09,-9.533493,-0.443493,7,tyk2,1.0,./A_QSAR_MIF/tyk2/PLS_Analysis/CV_Results_res_...,12198,0.001,O=C(Nc1ccnc(NC(=O)C2CCCC2)c1)c1c(Cl)cccc1Cl,322,ejm_48,0_DATA/COMPOUNDS/SDF/tyk2_lig_ejm_48.sdf
12198,99,-9.88,-9.952377,-0.072377,1,tyk2,1.0,./A_QSAR_MIF/tyk2/PLS_Analysis/CV_Results_res_...,12199,0.001,CCC(=O)Nc1cc(NC(=O)c2c(Cl)cccc2Cl)ccn1,316,ejm_42,0_DATA/COMPOUNDS/SDF/tyk2_lig_ejm_42.sdf


In [2]:


# ensure split sorted with 0 first
splits_sorted = sorted(predictions_df["split"].unique(), key=lambda x: (x != 0, x))

json_list = []
for split_val in splits_sorted:
    split_df = predictions_df[predictions_df["split"] == split_val]
    split_dict = {}
    for target in sorted(split_df["target"].unique()):
        target_df = split_df[split_df["target"] == target]
        target_dict = {}
        for _, row in target_df.iterrows():
            ligand = f"lig_{row['ligand']}"
            target_dict[ligand] = {
                "prediction": float(row["Predicted"]),
                "experimental": float(row["Actual"])
            }
        split_dict[target] = target_dict
    json_list.append(split_dict)

# save JSON
out_path = os.path.join(out_dir,"dG_pred_&_error_all_splits.json")
with open(out_path, "w") as f:
    json.dump(json_list, f, indent=2)
